In [2]:
import os
import sys
sys.path.append("..")  # add project root
import scipy.io as sio
import importlib
from prog import mlps, featlib, trainer, hlprs
importlib.reload(mlps)
import Datasets.load_mat_as_flat as lmf
import Datasets.matconv as mc
import torch
import matplotlib.pyplot as plt
import numpy as np

## Checking that data satisfies the Allen Cahn Equation with the correct parameters by finite difference

### On generated mat file

In [30]:
data = sio.loadmat("../Datasets/data/raw/Allen_Cahn_generated.mat")
u = data["u"] # data has shape (101, 201), which is (time, space)
t = data["t"] # time vector of shape (1, 101)
x = data["x"] # space vector of shape (1, 201)

dt = t[0,1] - t[0,0] # time step
dx = x[0,1] - x[0,0] # space step
u_xx = np.zeros_like(u)
u_t = np.zeros_like(u)

u_xx[1:-1, 1:-1] = (u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]) / (dx**2)

u_t[1:-1, 1:-1] = (u[2:, 1:-1] - u[:-2, 1:-1]) / (2*dt)


u_mid = u[1:-1, 1:-1] # values of u at the interior points
u_t = u_t[1:-1, 1:-1]
u_xx = u_xx[1:-1, 1:-1]
res = np.linalg.norm(u_t - 0.001*u_xx - 5*u_mid + 5*(u_mid)**3) # should be close to zero if the PDE is satisfied
print("RMS residual:", np.sqrt(np.mean(res**2)))
print("Relative residual:", np.linalg.norm(res) / np.linalg.norm(u_t))

RMS residual: 2.287849165450404
Relative residual: 0.017515619855685925


### On DeepXDE mat file

In [6]:
data = sio.loadmat("../Datasets/data/raw/Allen_Cahn.mat")
u = data["u"] # data has shape (101, 201), which is (time, space)
t = data["t"] # time vector of shape (1, 101)
x = data["x"] # space vector of shape (1, 201)

dt = t[0,1] - t[0,0] # time step
dx = x[0,1] - x[0,0] # space step
u_xx = np.zeros_like(u)
u_t = np.zeros_like(u)

u_xx[1:-1, 1:-1] = (u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]) / (dx**2)

u_t[1:-1, 1:-1] = (u[2:, 1:-1] - u[:-2, 1:-1]) / (2*dt)

u_mid = u[1:-1, 1:-1] # values of u at the interior points
u_t = u_t[1:-1, 1:-1]
u_xx = u_xx[1:-1, 1:-1]
res = u_t - 0.001*u_xx - 5*u_mid + 5*(u_mid)**3
print("RMS residual:", np.sqrt(np.mean(res**2)))
print("Relative residual:", np.linalg.norm(res) / np.linalg.norm(u_t)) # should be close to zero if the PDE is satisfied

RMS residual: 0.005742068597945677
Relative residual: 0.006151641100309764
